# <center>HermesAnalytics 第一节课：安装、架构与问数流程</center>

&emsp;&emsp;HermesAnalytics 是一套真实电商经营分析场景中的可信自然语言问数系统。它不是一个“模型随手写 SQL”的演示，而是一个把安装、架构、两个工作台、人工确认和数据库执行串成一条完整链路的工程项目。我们这节课要回答的问题很简单：当用户用一句中文问净销售额时，系统每一步由谁负责、页面能看到什么、源码在哪证明、什么证据才算真的跑通。

&emsp;&emsp;这节课我们重点抓三件事。第一件，把 `HermesAnalytics` 在本地跑起来，并建立一套分清“容器启动、ready、模型可用、完整 E2E”的**证据分级**。第二件，看懂前后端、Hermes Agent、Compiler、Safe Executor 和数据库之间的**系统责任边界**。第三件，沿唯一主案例走通分析工作台与 SQL 工作台，再回到可信 NL2SQL 的**完整问数主线**。

&emsp;&emsp;学习方法也很直接：先看界面和状态，再回源码找责任边界。我们不会堆大量代码，源码只作为证据入口；你只要知道每一步能证明什么、失败时先去哪里看。

> **目标受众与前置要求**：本课面向已有 Agent、Tool Calling、SQL、关系数据库和 HTTP API 基础的学员。你不需要预先了解 Hermes 或 NL2SQL，也不需要先安装 Python、Node、pnpm、Bun 或本地 PostgreSQL；本机只需 Docker 与 Compose v2。

> **学完本节你将带走这些能力**：① **安装证据**：区分容器状态、readiness、模型可用和当前 E2E；② **项目架构**：说清四服务、Hermes Agent 与前后端连接；③ **已有业务内容**：说清指标、维度和主案例净销售额口径；④ **两个工作台用法**：完成分析工作台操作并区分自由 SQL 与会话联动 SQL；⑤ **跨平台完整问数**：在人工确认点区分直接执行和条件式联动；⑥ **可信状态链**：复述状态、人工确认和系统职责。

> **本课范围与边界**：本课重点为安装、项目架构、分析工作台、SQL 工作台、唯一主案例和可信边界；归因工作台和管理员变更本节只介绍全貌，不展开。容器启动、readiness、模型可用和完整 E2E 是四类不同证据，不能混为一谈。

> **时效与证据边界**：本课唯一授课源码是 `HermesAnalytics`，源码事实与相对链接均指向该版本。课文中引用“HermesAnalytics 实际界面”时，表示该截图来自本机对 HermesAnalytics 界面做出的真实实测；引用“源码核验”时，表示该结论来自 HermesAnalytics 源码；尚未在本机验证的内容会明确标注“尚未验证”。历史记录只作参照，不能替代当前实测。

&emsp;&emsp;下面用一张路线表固定全课顺序和唯一主案例。

<p style="text-align: center;"><strong>表 0-1　课程路线与唯一主案例</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 章节 | 这一段要回答的问题 | 主案例位置 |
| --- | --- | --- |
| 课程导入 | 本课只追哪一条问数主线？ | 固定问题与路线 |
| 第一章：安装与启动 | 怎样取得源码、启动并判断服务？ | 尚未证明问数成功 |
| 第二章：项目架构 | 系统由谁组成，已经能分析什么？ | 建立架构与业务地图 |
| 第三章：分析与 SQL 工作台 | 本课重点的两个工作台分别怎样使用？ | 学会页面功能与基本操作 |
| 第四章：主案例的业务数据 | 主案例涉及哪些数据、字段与计算口径？ | 建立 SQL 正误判断标准 |
| 第五章：完整问数 | 怎样按证据梳理主案例两条合法路径？ | 梳理唯一主案例的两条路径 |
| 第六章：问数为什么可信 | 系统怎样限制模型并保护执行？ | 倒推可信原理 |
| 第七章：总结与自测 | 我能否复述并定位每个证据？ | 完成全课验收 |

</div>

&emsp;&emsp;唯一主案例始终是下面这一句。全课不增加第二主案例；后面问数章节会看到历史结果形态，只作参照，不能替换当前实测。

> **请分析 2026 年 6 月各渠道的净销售额，并与 2026 年 5 月比较。**

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170237730.png" alt="七章课程全景" width="100%"></p>

## <center>第一章：安装与启动</center>

&emsp;&emsp;本节只要求一台具备 Docker 与 Compose v2 的开发机；不需要预先安装 Python、Node、pnpm、Bun、uv 或本地 PostgreSQL。浏览器用于观察产品，但不替代终端证据。

### 1.1 前置条件

&emsp;&emsp;我们先看本课唯一必需环境：Docker Desktop（或等价 Docker Engine）与 `docker compose` v2。课程源码包或我们拿到的真实仓库地址是入口；模型凭据由你填写，不写入讲义或截图。Hermes Agent 已随交付源码进入 backend 构建流程，不需要提前单独安装。

&emsp;&emsp;开篇已经固定唯一主案例；本章才把课程源码变成可观察的运行栈。路线是：获取项目 → 认识四服务与预检边界 → 两次启动并填写模型变量 → 观察 `docker compose ps --all` 与 readiness → 区分当前观察和历史记录。

&emsp;&emsp;本章核心问题是：**每一道部署证据恰好说明什么，又不说明什么**？部署命令必须先于服务状态和 readiness；容器状态、ready/readiness、模型真实调用和主案例 E2E 分别回答不同问题。

### 1.2 安装并验证 Docker Desktop

&emsp;&emsp;本课使用 Docker Desktop 作为 macOS 与 Windows 的统一入口。它同时提供三样东西：Docker Engine（负责创建和运行容器的后台服务）、Docker CLI（在终端输入 `docker` 命令的命令行工具）和 Docker Compose，因此不需要再单独安装旧式的 `docker-compose`。安装页面与系统要求会随 Docker Desktop 更新，开始前请以官方文档为准：

- macOS：[Install Docker Desktop on Mac](https://docs.docker.com/desktop/setup/install/mac-install/)
- Windows：[Install Docker Desktop on Windows](https://docs.docker.com/desktop/setup/install/windows-install/)
- Compose 安装说明：[Overview of installing Docker Compose](https://docs.docker.com/compose/install/)

&emsp;&emsp;在 macOS 上，先确认芯片类型，再选择 **Apple silicon** 或 **Intel** 对应的安装包；完成安装后从“应用程序”启动 Docker Desktop。在 Windows 上，本课运行的是 Linux containers，优先使用 Docker Desktop 的 **WSL 2** 路径；如果单位电脑受管理员策略、虚拟化设置或代理限制，应先解决这些宿主机前置，再进入项目部署。

&emsp;&emsp;安装完成不等于 Docker 守护进程已经可用。等待 Docker Desktop 显示运行后，在下面的 Notebook 代码单元中执行两项检查：第一项确认客户端和服务端能通信；第二项确认本课使用的 `docker compose` 子命令可用。代码行开头的 `!` 表示由 Jupyter 调用系统命令行，而不是把这行当作 Python 语句。

In [ ]:
# 验证客户端与守护进程能通信
!docker version

# 验证当前环境使用 Compose v2
!docker compose version

&emsp;&emsp;首次执行 `./scripts/bootstrap.sh` 时，Docker 会自动拉取本机缺失的基础镜像，不需要逐个手动预检或拉取。如果网络不能直接访问镜像仓库，请先开启可用的代理/VPN 网络，并确认 Docker Desktop 自己能够使用这条网络路径；浏览器或终端能联网，并不代表 Docker Engine 可以拉取镜像。Docker Desktop 的代理应在 Desktop 设置中配置，而不是写入项目 `.env`。[Docker Desktop 代理设置](https://docs.docker.com/desktop/settings-and-maintenance/settings/)

<p style="text-align: center;"><strong>表 1-1　Docker 安装验证的结果判读</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 现象 | 说明 | 下一步 |
| --- | --- | --- |
| `docker version` 同时出现 Client 与 Server | Engine 已经响应 | 继续检查 Compose |
| 只有 Client 或提示无法连接 daemon | Desktop 未启动、未就绪或当前 context 不正确 | 打开 Desktop，等待就绪后重试 |
| `docker compose version` 正常输出 | Compose v2 可用 | 进入项目目录准备 |
| 只有 `docker-compose` 可用 | 当前环境仍依赖旧入口 | 不继续本节，先按官方说明补齐 Compose v2 |
| 构建命令拉取基础镜像失败 | Docker Desktop 未能访问镜像仓库 | 先配置 Desktop 的代理/VPN 网络，再重试同一启动命令 |

</div>

&emsp;&emsp;Docker 验证通过只证明 Engine 与 Compose v2 可用；1.5～1.9 继续检查项目启动、容器状态和 readiness，模型真实调用与主案例 E2E 需要在后续实际请求中分别验证，主案例完整链路在第五章展开。

### 1.3 获取项目

&emsp;&emsp;先判断你拿到的是哪种课程源码。本课优先使用课程源码包，因为它已经包含 `third_party/hermes-agent` 固定源码，不需要再执行 Git 子模块初始化；只有课程明确提供真实 Git URL 时，才使用 clone 作为可选路径。两种方式都以出现 `compose.yaml` 和 `scripts/bootstrap.sh` 为通过条件。

&emsp;&emsp;**操作目的：拿到课程源码包后，进入实际解压目录并核对部署入口。**

&emsp;&emsp;下面的命令直接在 Notebook 代码单元中执行。代码行开头的 `!` 表示由 Jupyter 调用系统命令行；请把尖括号中的占位内容替换为真实路径后，按原 Bash 命令顺序执行。

In [ ]:
# 进入课程源码包的实际解压目录
!cd <课程源码包实际解压目录>

# 核对 Compose 与启动脚本
!ls compose.yaml scripts/bootstrap.sh

&emsp;&emsp;预期输出是两条文件路径。它能证明你位于含有 Compose 编排和启动脚本的项目根目录；不能证明 Docker 可用、服务能启动或模型已经配置。若失败，优先检查你是否只进入了外层压缩包目录、文件是否解压完整、目录名是否被自行猜错。

&emsp;&emsp;**可选路径：只有课程明确提供真实 Git URL 时，才执行下面的 clone 命令。**

In [ ]:
# 克隆课程提供的真实仓库
!git clone <你拿到的真实仓库地址>

# 进入 clone 后的实际目录
!cd <git clone 后的实际目录名>

# 核对 Compose 与启动脚本
!ls compose.yaml scripts/bootstrap.sh

&emsp;&emsp;预期输出同样是 `compose.yaml` 与 `scripts/bootstrap.sh`。它能证明 Git 已按课程给出的地址取得一个含部署入口的目录；不能证明该地址可替代源码包、Docker 已就绪或任何服务已运行。若失败，优先看课程给出的 URL、网络/凭证、clone 后的真实目录名；本课不发明仓库 URL。

### 1.4 启动前检查

&emsp;&emsp;`./scripts/bootstrap.sh --dry-run` 可以作为源码/计划预检：它检查交付包中的 Hermes 固定源码并打印计划，但不会启动 Docker，也不会写入运行结果。它出现 error 或 warning 时应先观察原因；即使 exit code 为 0，也不等于容器、readiness、模型调用或端到端 E2E 成功。

&emsp;&emsp;本课的正式操作链不以 dry-run 代替启动。若你选择额外运行它，请把它当作“我准备执行什么”的检查，而不是“我已部署成功”的凭证。

&emsp;&emsp;常见误解是“容器都起来了，所以模型一定能问数”。真实机制是递进证据：容器状态、ready/readiness、模型真实调用、主案例 E2E 分别回答不同问题。`ready/readiness` 在本课指应用声明其关键依赖已就绪的 HTTP 检查，不等于模型调用已成功；E2E 指从用户入口到结果与证据的完整端到端路径。

### 1.5 第一次启动

&emsp;&emsp;**操作目的：让脚本生成或保留 `.env`，检查 Compose 配置，构建并等待服务。**

In [ ]:
# 执行项目原始启动脚本
!./scripts/bootstrap.sh

&emsp;&emsp;预期输出会说明环境文件被创建或保留，Compose 构建、启动并等待服务，最后给出类似 `http://localhost:8080` 的入口。它能证明本次 Docker/Compose 启动链已经完成脚本要求的等待；不能证明模型凭据已正确填写、模型提供商可调用或唯一主案例已经跑通。若失败，优先看第一条 error、Docker daemon、磁盘/镜像构建、`.env.example` 与脚本输出中的具体服务名，而不是只看最后一行。

### 1.6 配置模型

&emsp;&emsp;第一次启动后，打开项目根目录的 `.env`，只填写或核对下列四项模型接入变量。不要把 Key 写入讲义、截图、终端回显或提交记录；演示账号密码由脚本处理，模型变量与随机本地密码是两类配置。

```text
HERMES_ANALYTICS_MODEL_PROVIDER=
HERMES_ANALYTICS_MODEL_NAME=
HERMES_ANALYTICS_MODEL_API_KEY=
HERMES_ANALYTICS_MODEL_BASE_URL=
```

&emsp;&emsp;这个 text 块只列出变量名，不提供某一家厂商的猜测值。填写文件能证明“模型配置有明确入口”，不能证明凭据正确或远端模型可用。失败或不确定时，先检查课程随附的模型接入说明与变量拼写，绝不把 Key 粘贴到公开位置。

### 1.7 第二次启动

&emsp;&emsp;**操作目的：在已填写模型变量后，重新按脚本启动，使当前容器读取新的 `.env`。**

In [ ]:
# 重新启动以加载最新模型配置
!./scripts/bootstrap.sh

&emsp;&emsp;预期输出仍是 Compose 构建/启动/等待和访问地址。它能证明脚本按当前 `.env` 重走启动过程；不能仅凭脚本退出成功就证明模型请求已到达提供商。若失败，先看 `.env` 格式、Docker 输出、backend 的首条 error；不要用 `--dry-run` 代替这一步，因为 dry-run 不启动服务。

### 1.8 查看服务状态

&emsp;&emsp;**操作目的：查看四个服务当前的容器状态。**

In [ ]:
# 查看运行中与已退出的全部服务
!docker compose ps --all

&emsp;&emsp;预期是 `postgres`、`backend`、`frontend` 处于运行状态，`migrate` 作为一次性初始化任务完成后退出。它能证明 Compose 层此刻报告的服务状态；不能证明前端反代、业务 API、模型调用和主案例完整 E2E。若失败，优先看服务是否为 `Exited`、健康检查是否失败、`docker compose logs <服务名>` 的第一条报错。

### 1.9 检查服务就绪

&emsp;&emsp;**操作目的：我们通过产品用户入口检查应用 readiness。**

In [ ]:
# 同时观察 HTTP 状态与响应正文
!curl -i http://localhost:8080/api/v1/health/ready

&emsp;&emsp;通过标准必须同时满足：HTTP 状态为 **200**，并且响应中的 `components` 满足 **READY（就绪）**。任意 HTTP 响应都不算通过，例如 502 只是“前端收到一个错误响应”。它能证明该时点的应用依赖已达到 ready 条件；不能证明模型可回答、人工确认已发生或主案例 E2E 已完成。失败时优先看状态码、`components` 中哪个组件非 READY（未就绪）、frontend 到 backend 的反代，以及相关容器日志。

<p style="text-align: center;"><strong>表 1-2　证据分级的严格边界</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 证据分级 | 能证明 | 不能证明 |
| --- | --- | --- |
| 容器运行/healthy | 编排与容器健康检查在该时点成立 | 模型可用、业务问数成功 |
| readiness 200 + components READY（就绪） | 应用依赖达到 ready 条件 | 模型一定可调用、主案例已完成 |
| 模型真实可调用 | 当前模型配置沿产品路径请求成功 | 唯一主案例的确认、执行与证据都成功 |
| 主案例 E2E | 用户提交、确认、执行、结果校验、证据和解读完成可追溯结果 | 未来所有环境必然成功 |

</div>

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170237664.png" alt="证据分级" width="85%"></p>

### 1.10 浏览器登录

1. **你要做的操作**：在浏览器打开 `http://localhost:8080`，使用 `.env` 中的演示分析师账号登录，并进入分析工作区。
2. **界面/API 可见状态**：登录成功后进入分析师产品界面；历史记录中登录 API 为 HTTP 200、角色为 `DATA_ANALYST`（数据分析师）。
3. **谁负责**：frontend 负责界面与同源请求；backend 负责认证与会话；你负责输入凭据。
4. **证据等级**：你本机实际登录后属于本机实测；这里提及的历史 API 结果仅为历史运行快照。
5. **通过标准**：你能看到已登录的分析工作区，而不是只看到 HTTP 接口或静态图片。失败时先看登录错误、浏览器网络面板和 backend 认证日志。

&emsp;&emsp;这张截图来自 HermesAnalytics 实际界面，展示浏览器登录页。看什么：登录入口、演示账号输入位置和登录按钮。能证明什么：HermesAnalytics 前端登录链路在当前环境可到达，并已取得真实页面截图；不能证明模型问数或完整 E2E 已成功。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170259757.png" alt="浏览器登录页" width="95%"></p>

### 1.11 区分两类证据

&emsp;&emsp;当前运行证据以本机对 HermesAnalytics 界面做出的实测为准；历史记录只作参照，不能替代当前结果。它记录了历史时点的容器状态、readiness、浏览器路径 A/追问/路径 B/自由 SQL 边界，以及模型配置与重建风险。历史观察中的容器退出、readiness 与浏览器 E2E 未验证等内容，属于历史证据，不能借用历史结果替代当前状态；你的本机状态以 HermesAnalytics 重新实测为准。

&emsp;&emsp;历史记录包含早期的 502 恢复、API 主案例、工作台联动和静态 PNG，以及后来对 HermesAnalytics 界面的浏览器核查。读取时先看章节标题，历史与阶段内容不得冒充 HermesAnalytics 当前状态；当前浏览器和 API 证据以 HermesAnalytics 实际界面为准。

## <center>第二章：项目架构</center>

&emsp;&emsp;项目已经启动，下一步先看系统由哪些部分组成，以及当前领域里已经准备了哪些业务内容。本章只建立架构与业务地图，不展开可信问数内部字段。

### 2.1 四个服务

&emsp;&emsp;项目把浏览器入口、后端、迁移和数据库拆成四个服务。浏览器只从 `frontend` 的 `8080` 进入；`backend` 和 PostgreSQL 都不是用户直接访问的入口。

<p style="text-align: center;"><strong>表 2-1　Compose 服务的职责</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 服务 | 主要职责 | 用户是否直接访问 |
| --- | --- | --- |
| `postgres` | 保存业务数据、领域版本、会话、结果和审计数据 | 否 |
| `migrate` | 一次性建角色、建表、灌教学数据并准备默认领域内容 | 否；完成后退出 |
| `backend` | FastAPI、认证、Hermes 适配、可信 NL2SQL、安全执行与结果服务 | 否；通过前端反代访问 |
| `frontend` | React 页面与 `/api` 反代 | 是，默认端口 `8080` |

</div>

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170338379.png" alt="部署图" width="85%"></p>

### 2.2 技术框架

&emsp;&emsp;2.1 已经看到项目由四个 Compose 服务组成。现在再进一步，看每个服务内部用什么技术，以及一次请求如何穿过前端、后端、Agent 和数据库。学完本节，你可以从“部署服务”和“技术层次”两个角度读懂 HermesAnalytics 的整体结构。

&emsp;&emsp;先拆四个最容易出现的误解。

&emsp;&emsp;**误解一：React 就是前端服务，Hermes Agent 是第五个 Compose 服务。** 真实机制：React 负责浏览器页面，Vite 负责开发和构建；生产环境由前端容器里的 Nginx 提供静态页面并反向代理 `/api`。Hermes Agent 不是独立服务，而是 backend 镜像内安装的 Python 包。

&emsp;&emsp;**误解二：SQLGlot 负责生成 SQL。** 真实机制：SQLGlot 主要做 SQL 解析、规范化和 AST 安全策略检查；主案例的 SQL 由项目确定性编译逻辑生成，SQLGlot 不负责写 SQL。

&emsp;&emsp;**误解三：后端业务查询都走 SQLAlchemy ORM。** 真实机制：业务数据库访问主要通过 Psycopg 异步连接池；SQLAlchemy 主要用于在迁移和重置工具中表达、操作数据库结构，Alembic 负责组织、执行和追踪迁移版本，不能把二者笼统讲成后端业务 ORM。

&emsp;&emsp;**误解四：uv 和 pnpm 是学员本机必须安装的运行环境。** 真实机制：uv 和 pnpm 主要在镜像构建阶段使用；按照第一章的前置要求，学员本机只需要 Docker 与 Compose v2。

&emsp;&emsp;一句话定义：HermesAnalytics 的技术框架 = React 前端应用 + Nginx 访问入口 + FastAPI 后端 + Hermes Agent 与确定性问数程序 + PostgreSQL 三层数据库 + Docker Compose 部署。前端构建产物由 Nginx 提供，浏览器只访问 8080；backend 在内部 8000 提供 API，PostgreSQL 不直接对外。

&emsp;&emsp;判断边界：React/Vite 负责“开发与构建”，Nginx 负责“生产静态资源与 /api 反代”；Hermes Agent 属于“backend 内部模型运行组件”，不是第五个服务；SQLGlot 属于“安全与解析组件”，不是 SQL 生成器；Psycopg 负责业务数据访问，SQLAlchemy 主要在迁移和重置工具中表达、操作数据库结构，Alembic 负责组织、执行和追踪迁移版本。这些边界不搞清楚，后面读源码时容易找错文件。

&emsp;&emsp;下面用一张技术框架图建立空间关系。它只讲技术分层，不替代 2.6 的可信问数链路图。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170244527.png" alt="HermesAnalytics 技术框架分层图" width="150%"></p>


&emsp;&emsp;课堂按三步读图：先看浏览器、Nginx 和 React 的入口层；再看 FastAPI、Hermes Agent 与确定性程序的 backend 内部；最后看 Psycopg、PostgreSQL 和底部工程支撑。表 2-2 作为读完图后的源码索引，不逐项念技术名词。

&emsp;&emsp;这张图是“技术框架分层图”，不是 2.6 的可信问数链路图。它只回答系统用什么技术、各层如何连接；模型如何规划、编译、执行，后续章节再展开。

&emsp;&emsp;下面是新增的表 2-2，把技术层的课堂重点固定下来。

<p style="text-align: center;"><strong>表 2-2　技术框架与职责</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 技术层 | 核心技术 | 在项目中的职责 | 源码锚点 |
| --- | --- | --- | --- |
| 前端页面层 | React、TypeScript、React Router | 构建数据分析师与数据库管理员页面，路由区分工作台和后台 | [`frontend/package.json`](../../HermesAnalytics/frontend/package.json)、[`frontend/src/app/router.tsx`](../../HermesAnalytics/frontend/src/app/router.tsx) |
| 前端支撑层 | Vite、Tailwind CSS、TanStack Query、Zustand、ECharts、CodeMirror | Vite 负责构建；其余负责样式、请求状态、主题、图表和 SQL 编辑 | [`frontend/vite.config.ts`](../../HermesAnalytics/frontend/vite.config.ts)、[`frontend/src/shared/state/themeStore.ts`](../../HermesAnalytics/frontend/src/shared/state/themeStore.ts) |
| 前端访问层 | Nginx | 生产环境提供静态页面，并把 `/api` 反向代理到 backend | [`frontend/Dockerfile`](../../HermesAnalytics/frontend/Dockerfile)、[`deploy/nginx.conf`](../../HermesAnalytics/deploy/nginx.conf) |
| 后端 API 层 | Python 3.12、FastAPI、Uvicorn、Pydantic | 提供认证、分析、SQL 工作台、归因和管理员 API；Pydantic 负责请求与领域对象校验 | [`backend/pyproject.toml`](../../HermesAnalytics/backend/pyproject.toml)、[`backend/src/hermes_analytics/main.py`](../../HermesAnalytics/backend/src/hermes_analytics/main.py) |
| 智能与确定性处理层 | Hermes Agent、Query Planner、SQL Compiler、SQLGlot、Freeze | Hermes 负责模型调用；程序负责规划、编译、策略检查和冻结 | [`backend/src/hermes_analytics/hermes_adapter/agent.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/agent.py)、[`backend/src/hermes_analytics/nl2sql/policy/inspector.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/policy/inspector.py) |
| 数据访问层 | Psycopg、SQLAlchemy、Alembic | Psycopg 负责业务连接；SQLAlchemy 在迁移/重置工具中操作数据库结构；Alembic 管理迁移版本 | [`backend/src/hermes_analytics/infrastructure/postgres/pool.py`](../../HermesAnalytics/backend/src/hermes_analytics/infrastructure/postgres/pool.py)、[`database/migrations/env.py`](../../HermesAnalytics/database/migrations/env.py)、[`database/hermes_analytics_database/reset.py`](../../HermesAnalytics/database/hermes_analytics_database/reset.py) |
| 数据层 | PostgreSQL 17 | 保存业务数据、治理视图和系统数据，区分 `ecommerce_core`、`analytics`、`system` | [`compose.yaml`](../../HermesAnalytics/compose.yaml)、[`database/migrations/versions/20260731_0001_initial_schema.py`](../../HermesAnalytics/database/migrations/versions/20260731_0001_initial_schema.py) |
| 部署层 | Docker Compose、Alembic、uv、pnpm | Compose 负责服务编排和网络边界；Alembic 负责迁移；uv/pnpm 在镜像构建阶段使用 | [`compose.yaml`](../../HermesAnalytics/compose.yaml)、[`backend/Dockerfile`](../../HermesAnalytics/backend/Dockerfile)、[`frontend/Dockerfile`](../../HermesAnalytics/frontend/Dockerfile) |

</div>

&emsp;&emsp;版本说明：本课只锁定课堂必要的主版本——后端 Python 3.12、前端 React 19、数据库 PostgreSQL 17。具体补丁版本以源码锁文件为准，课件不逐一列出。

&emsp;&emsp;一条请求路径：浏览器打开 `http://localhost:8080` → Nginx 提供前端页面 → 页面调用同源 `/api/v1/...` → Nginx 反向代理到 backend 的 `8000` → FastAPI 接收请求 → backend 内部按业务调用 Hermes Agent 或确定性问数程序 → 需要数据时通过 Psycopg 连接池访问 PostgreSQL → 后端返回 API 响应 → 页面更新结果。这条路径中，backend 8000 和 PostgreSQL 不直接暴露给外部浏览器，前端是唯一入口。

&emsp;&emsp;口语类比：前端是门店柜台，Nginx 是门卫和叫号机，FastAPI 是业务前台，Hermes Agent 是请来的专家顾问，确定性问数程序是内部合规流程，Psycopg 是传话通道，PostgreSQL 是保险柜。边界例：专家顾问不能直接进保险柜拿数据，必须走内部合规流程和传话通道；门卫也不能自己当业务前台，只负责接待和转交请求。

&emsp;&emsp;通过标准：你能用一句话说清前端、Nginx、FastAPI、Hermes Agent、Psycopg 和 PostgreSQL 各自负责什么，并判断 Hermes Agent 不是第五个服务、SQLGlot 不生成 SQL、SQLAlchemy/Alembic 不是业务 ORM。

### 2.3 Hermes Agent

&emsp;&emsp;2.2 已经确认 Hermes Agent 位于 backend 内部。这里继续回答两个问题：这份 Python 包怎样进入 backend 镜像，以及项目怎样封装并调用它。学员本机只需 Docker 与 Compose v2，不需要执行 `pip install` 或 `uv sync`，也不要在本机额外安装 Hermes Agent。

&emsp;&emsp;交付包已经在 `third_party/hermes-agent` 中带有固定源码，`.gitmodules` 将该目录指向 `https://github.com/NousResearch/hermes-agent.git`。正常启动（非 dry-run）时，如果该目录缺失，并且当前目录是 Git 仓库、机器上也有 Git，[`scripts/bootstrap.sh`](../../HermesAnalytics/scripts/bootstrap.sh) 会自动初始化子模块；dry-run 只做预检，不会初始化子模块。

&emsp;&emsp;启动脚本会核对源码版本和 `deploy/HERMES_AGENT_COMMIT`；随后 [`backend/Dockerfile`](../../HermesAnalytics/backend/Dockerfile) 把这份源码复制进 backend 镜像，并在镜像构建中完成安装。当前课程基线是 Hermes Agent `0.19.0`，固定提交记录在 `deploy/HERMES_AGENT_COMMIT` 中。[`backend/pyproject.toml`](../../HermesAnalytics/backend/pyproject.toml) 也锁定 `hermes-agent==0.19.0` 并指向这份本地源码。

In [ ]:
# 查看 Hermes Agent 包版本
!grep '^version = ' third_party/hermes-agent/pyproject.toml

# 查看课程冻结的源码提交
!cat deploy/HERMES_AGENT_COMMIT

&emsp;&emsp;预期分别看到 `version = "0.19.0"` 和冻结提交标识。它们只能证明源码基线，不能证明镜像构建或问数 E2E 已成功。

&emsp;&emsp;说完包在哪，下一步看这个包怎么被调用。Hermes Agent 的 Python 入口在 [`third_party/hermes-agent/run_agent.py`](../../HermesAnalytics/third_party/hermes-agent/run_agent.py) 的 `AIAgent` 类，`run_conversation()` 是它的对话入口。但项目生产链路并不是业务代码到处直接创建 `AIAgent`，而是把它再包了一层，形成一条固定链路：`AIAgent` → `HermesAnalyticsAgent` → `HermesAgentFactory` → `BoundedHermesRunner`。我们先把这条链路讲清楚，再做一个最小真实调用。

<p style="text-align: center;"><strong>表 2-3　Hermes Agent 包调用链路</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 环节 | 在源码里的位置 | 负责什么 |
| --- | --- | --- |
| 包入口 | [`third_party/hermes-agent/run_agent.py`](../../HermesAnalytics/third_party/hermes-agent/run_agent.py) | 提供 `AIAgent` 类与 `run_conversation()` 对话方法，返回 `dict` |
| 项目封装 | [`backend/src/hermes_analytics/hermes_adapter/agent.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/agent.py) | `HermesAnalyticsAgent` 锁定上下文、迭代边界与工具白名单 |
| 工厂 | [`agent.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/agent.py) 的 `HermesAgentFactory` | 每次 operation 创建全新 Agent，按阶段锁死工具集合 |
| 运行器 | [`backend/src/hermes_analytics/hermes_adapter/runner.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/runner.py) 的 `BoundedHermesRunner` | 在线程池承载同步 Hermes，负责超时与协作式中断 |

</div>

&emsp;&emsp;先看包本身的调用方式。`AIAgent` 的真实构造参数包括 `model`、`provider`、`base_url`、`api_key`；`run_conversation()` 接受 `user_message` 与 `system_message`，返回值是 `dict`，课堂只读取 `final_response` 与 `api_calls` 两个字段。`api_calls` 记录本次对话的 Agent 外层模型调用计数，`final_response` 是模型最终回复文本。

&emsp;&emsp;再看项目封装。`HermesAnalyticsAgent` 强制每次调用都提供系统提示词，并把 `skip_context_files`、`skip_memory`、`save_trajectories` 设为固定值，把 `max_iterations` 限制为阶段预算；`HermesAgentFactory` 每次 operation 创建全新 Agent，并按阶段锁定工具白名单；`BoundedHermesRunner` 在线程池中运行同步 Hermes，超时后先调用 `agent.interrupt()` 协作式中断，再按宽限期回收。这就是后续主链会展开的“包可用”与“业务链路可用”之间的边界：能直接调用 `AIAgent`，不等于 HermesAnalytics 的规划、确认、编译与执行链路已经跑通。

&emsp;&emsp;在往下做最小调用之前，先把一个最容易混的边界讲清楚：HermesAnalytics 到底保留了 Hermes Agent 的什么、未启用或未暴露什么、哪些是项目自己实现的。先拆四个常见误解。

&emsp;&emsp;**误解一：引入 Hermes Agent 就等于启用 Hermes 的全部个人助理能力。** 真实机制：项目只把它当作受控的模型运行内核，terminal/browser/file/subagent/cron/原生 skills 等未进入当前阶段白名单的通用能力不会暴露给当前阶段业务 Agent。

&emsp;&emsp;**误解二：`skip_memory=True` 就等于项目不能多轮追问。** 真实机制：`skip_memory` 关闭 Hermes 内置记忆加载；项目自己的多轮连续性由 PostgreSQL 保存最近完整轮次并显式传入 `conversation_history`。

&emsp;&emsp;**误解三：`ContextCompressor` 就是跨会话长期记忆。** 真实机制：它只控制当前 Agent 调用窗口内的上下文压缩，不负责跨 operation 的业务持久化。

&emsp;&emsp;**误解四：`backend/skills` 就是 Hermes 原生 skills toolset。** 真实机制：`backend/skills` 是项目管理员操作技能目录，项目自建 `list_skills`/`read_skill`，不注册 Hermes 原生 `toolset="skills"`。

&emsp;&emsp;一句话定位：**在 HermesAnalytics 中，Hermes Agent 被当作受控的模型运行内核使用。** 这是本课为理解项目关系给出的定位描述，不是 Hermes 官方术语。

&emsp;&emsp;下面这张三层边界表只列关键能力与主锚点。

<p style="text-align: center;"><strong>表 2-4　Hermes Agent 在 HermesAnalytics 中的保留与关闭边界</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 边界 | 关键内容 | 主源码锚点 |
| --- | --- | --- |
| Hermes 原生能力中项目未启用或未暴露的部分 | `skip_context_files=True`、`skip_memory=True`、`session_db=None`、`save_trajectories=False`；每阶段显式非空 `enabled_toolsets` 白名单，未进入当前阶段白名单的通用能力不暴露给当前阶段业务 Agent | [`agent.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/agent.py) 的 `HermesAnalyticsAgent` 与 `HermesAgentFactory._create` |
| 项目仍保留和使用的 Hermes 运行能力 | provider/model 调用与适配；`run_conversation` 的工具调用循环、工具结果回填和最终响应；显式 `conversation_history`；当前调用窗口内的 `ContextCompressor`；每阶段迭代预算、messages/api_calls；生产超时/取消由 `BoundedHermesRunner` 调用 `interrupt`，最小演示显式调用 `close` 清理 | [`adapter.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/adapter.py) 的 `_run`；[`runner.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/runner.py) 的 `BoundedHermesRunner`；[`agent_init.py`](../../HermesAnalytics/third_party/hermes-agent/agent/agent_init.py) 的 `ContextCompressor` 构造 |
| HermesAnalytics 自己实现的业务能力：业务状态与上下文 | PostgreSQL 会话、最近最多 6 个完整轮次、`current_context`/`last_intent`/`last_query_id` | [`analysis_planning_repository.py`](../../HermesAnalytics/backend/src/hermes_analytics/infrastructure/postgres/analysis_planning_repository.py) `load_planning_input`；[`context.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/context.py) `build_explicit_history` |
| HermesAnalytics 自己实现的业务能力：确定性问数与执行证据 | Intent → 编译/Policy/Freeze → 人工确认/执行 → 结果校验与证据（ResultSnapshot/ResultDigest/FactRef） | 架构铁律见项目 [`AGENTS.md`](../../HermesAnalytics/AGENTS.md)，机制展开见后续章节 |
| HermesAnalytics 自己实现的业务能力：项目管理员技能 | `backend/skills` 与自建 `list_skills`/`read_skill`，不注册 Hermes 原生 `toolset="skills"` | [`admin/plugin.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/admin/plugin.py) `install_admin_hermes_plugin` |

</div>

&emsp;&emsp;上下文传递有两条并行显式通道，最后汇合到本轮模型调用。**消息历史通道**：PostgreSQL `model_messages` → 最近最多 6 个完整轮次 → `build_explicit_history` → `HermesInvocation.history` → adapter 作为 `conversation_history` 传入。**业务上下文通道**：PostgreSQL `current_context` → `AnalysisContext(last_intent/last_query_id)` → `TurnContext.analysis_context` → 当轮 system prompt。另有 `history_candidates` 也是项目提供的显式输入，本课只需知道：最多 5 个历史查询候选会进入 turn context。结论：项目多轮连续性来自项目显式组装的消息历史与业务上下文，不是 Hermes Memory。

&emsp;&emsp;一句话类比：Hermes 是受控发动机，HermesAnalytics 提供车身、导航路线、交通规则和行车记录。正例：上一轮问“华东地区销售额”，下一轮只问“那环比呢？”，源码只证明项目会把可见历史显式传入、为新 Agent 承接指代提供上下文；是否正确理解仍需真实模型调用验证。真边界例：如果相关信息既不在可见消息历史中，也未通过 `current_context`、`history_candidates` 或其他当轮显式输入提供，就不能依靠 Hermes Memory 自动恢复；能否承接仍以本轮显式输入和真实模型调用为准，拿不准时走澄清。通过标准：学员能判断“历史显式传入”与“Hermes Memory 自动恢复”的区别。

&emsp;&emsp;边界判断：请判断四句话是否成立，并指出验证锚点——① 引入包=启用全部个人助理能力；② `skip_memory=True`=不能多轮追问；③ `ContextCompressor`=跨会话长期记忆；④ `backend/skills`=Hermes 原生 skills。

&emsp;&emsp;下面做一个最小真实模型调用。它不使用任何业务工具、不访问数据库、不走 NL2SQL，只证明一件事：当前 backend 容器里的 Hermes Agent 包能够用配置好的模型完成一次真实对话。命令在项目根目录执行，直接读取容器内已注入的 `HERMES_ANALYTICS_MODEL_*` 环境变量，不需要把密钥写进代码。

In [ ]:
# 在 backend 容器内运行原始验证脚本
!docker compose exec -T backend python - <<'PY'
import os

# 只读取容器内已注入的环境变量，不把密钥写进代码
provider = os.environ.get("HERMES_ANALYTICS_MODEL_PROVIDER", "")
model = os.environ.get("HERMES_ANALYTICS_MODEL_NAME", "")
base_url = os.environ.get("HERMES_ANALYTICS_MODEL_BASE_URL", "")
api_key = os.environ.get("HERMES_ANALYTICS_MODEL_API_KEY", "")
api_key_required = os.environ.get("HERMES_ANALYTICS_MODEL_API_KEY_REQUIRED", "true").strip().lower() != "false"
if not all([provider, model, base_url]) or (api_key_required and not api_key):
    raise SystemExit("模型配置不齐全，请先完成 .env 中的 HERMES_ANALYTICS_MODEL_* 配置")

from run_agent import AIAgent

agent = AIAgent(
    session_id="lesson1-package-demo",
    model=model,
    provider=provider,
    base_url=base_url,
    api_key=api_key or None,
    enabled_toolsets=[],  # 课堂演示不启用任何业务工具
    max_iterations=1,
    tool_delay=0,
    quiet_mode=True,
    skip_context_files=True,
    skip_memory=True,
    save_trajectories=False,
    session_db=None,
)

try:
    result = agent.run_conversation(
        "请用一句话解释数据分析为什么需要口径一致。",
        system_message="你是数据分析教学演示助手，回答保持简短。",
    )
    print("真实模型调用通过")
    print("final_response:", result.get("final_response"))
    print("api_calls:", result.get("api_calls"))
except Exception as exc:
    print("真实模型调用未通过:", type(exc).__name__, str(exc))
finally:
    agent.close()
PY

&emsp;&emsp;课堂执行时重点关注三处输出：`真实模型调用通过`、`final_response` 出现一句通顺解释、`api_calls` 为 `1`。`api_calls` 表示一次 Agent 外层迭代；结合 `max_iterations=1` 和未启用业务工具，本演示不会进入下一轮工具调用，但它不等于 Provider 重试次数，不能据此断言底层没有重试。如果输出 `真实模型调用未通过`，按顺序检查四件事：`.env` 中 `HERMES_ANALYTICS_MODEL_*` 变量是否齐全、provider 拼写、base_url 是否可达、模型名是否被当前提供商支持。这个检查只验证模型链路，不验证 HermesAnalytics 业务链路。

&emsp;&emsp;最后明确这段演示的证明边界。**能证明**：Hermes Agent 包可导入、真实 Provider 调用成功、返回结构可读取、`close()` 清理可执行。**不能证明**：HermesAnalytics 的结构化规划、人工确认、SQL 编译、安全执行或数据库 E2E。要看生产链路，锚点分别是 [`backend/src/hermes_analytics/hermes_adapter/agent.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/agent.py) 的 `HermesAnalyticsAgent`（源码第 81 行）与 `HermesAgentFactory`（源码第 231 行），以及 [`backend/src/hermes_analytics/hermes_adapter/runner.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/runner.py) 的 `BoundedHermesRunner`（源码第 18 行）。这些类在 HermesAnalytics 源码中真实存在，后续章节会沿它们继续展开。

### 2.4 项目特点

&emsp;&emsp;HermesAnalytics 面向电商经营分析，交付数据分析师和数据库管理员两个用户角色。两个角色的技术责任边界在 2.6 展开，这里先看各自使用或治理哪些功能。

&emsp;&emsp;**数据分析师**有三个工作台：分析工作台、SQL 工作台、归因工作台。第一节课只重点操作分析与 SQL 工作台；归因工作台和管理员变更链路只做全貌介绍，本课不展开。

&emsp;&emsp;**数据库管理员**在 `/admin` 下维护分析师赖以工作的三层内容：业务表的结构与数据、语义层的指标口径，以及 Hermes 做变更时遵循的方法。管理员变更链路是另一条独立链路，本课不演示管理员变更。

&emsp;&emsp;下面这张图把两个角色的功能放在同一张地图里。左侧是分析师使用的三个工作台；右侧是管理员负责的六个治理模块。中间箭头只表示治理内容支撑分析师使用，不表示查询或数据库执行流程。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170239722.png" alt="HermesAnalytics 双角色功能分工" width="95%"></p>


&emsp;&emsp;这张管理员总览截图来自 HermesAnalytics 实际界面，用于补足双角色整体认识。看什么：管理员端的管理总览、模型连接状态和变更入口。能证明什么：HermesAnalytics 确实交付了与分析师不同的管理员角色，管理员模型是独立配置；不能证明本课会演示管理员变更。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170235219.png" alt="管理员总览" width="95%"></p>

&emsp;&emsp;注意：截图中若显示“管理员模型密钥未配置”，这是当前真实 UI 状态，表示管理员链路还没有配置独立的 `HERMES_ANALYTICS_ADMIN_MODEL_*` 凭据。管理员模型与分析模型是两套独立配置，本课不演示管理员变更，所以不能把“管理员模型密钥未配置”误写成整个项目模型不可用。

### 2.5 系统能分析什么

- 业务范围：电商经营分析。
- 5 个指标：净销售额、支付订单数、支付买家数、客单价、退款金额率。
- 4 类维度：时间、渠道、商品、地区。
- 当前领域数据：6 个渠道、8 个品类、10 个地区。
- 主案例口径：净销售额 = 实付商品金额 − 按退款发生时间归属的退款商品金额。
- 主案例展开方式：按渠道比较 2026 年 6 月与 2026 年 5 月。
- 业务限制示例：退款金额率不支持商品维度，因为按订单归属的退款拆到商品粒度没有可靠业务意义。

&emsp;&emsp;为了让学员一眼看懂主案例包含哪些分析条件，可以把唯一主案例拆成五项：

<p style="text-align: center;"><strong>表 2-5　主案例问题拆解</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 教学拆解项 | 本案例取值 |
| --- | --- |
| 指标 | 净销售额 |
| 当前分析期 | 2026 年 6 月 |
| 对比期 | 2026 年 5 月 |
| 展开维度 | 渠道 |
| 分析动作 | 比较两个相邻月份各渠道的净销售额 |

</div>

&emsp;&emsp;这是帮助学员阅读自然语言问题的教学拆解，不代表模型已经提交运行时 Intent，也不代表 SQL 已执行；真正的结构化规划与状态链在后续章节讲解。

&emsp;&emsp;这些定义来自 Manifest。规则怎样校验、发布和切换版本，本课不展开。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170239751.png" alt="业务内容地图" width="85%"></p>

### 2.6 职责边界

&emsp;&emsp;浏览器访问 `frontend`，前端通过 `/api` 调用 FastAPI；backend 内部连接 Hermes Agent、可信 NL2SQL 核心和 Safe/Workbench Executor；执行器再访问 PostgreSQL。`migrate` 只负责一次性初始化。Hermes Agent 不直接连接数据库，也不拥有任意执行 SQL 的工具。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170244558.png" alt="本课聚焦的可信问数链路架构" width="150%"></p>


&emsp;&emsp;上面这张可信问数链路架构图展示分析师主案例从模型规划、程序编译、人工确认到执行器与数据库的边界；它不是 HermesAnalytics 全部功能架构。HermesAnalytics 还包含管理员变更链路、归因工作台等能力，本课只做整体认识，不展开内部字段。

### 2.7 前后端入口

&emsp;&emsp;前端业务路由集中在 [`frontend/src/app/router.tsx`](../../HermesAnalytics/frontend/src/app/router.tsx)，可以看到分析工作台、SQL 工作台与归因工作台的页面入口。后端由 FastAPI 提供认证、分析会话、确认执行和工作台 API。本课只沿分析与 SQL 工作台主入口阅读，不逐文件展开。

## <center>第三章：分析与 SQL 工作台</center>

&emsp;&emsp;第二章已经建立系统角色与模块边界。本章先让你会使用分析工作台和 SQL 工作台，知道问题从哪里进入、确认卡在哪里出现、三种 SQL 来源有什么区别。下一章再把确认卡中的查询还原到具体数据视图和字段，形成判断查询是否符合业务口径的依据。

&emsp;&emsp;先记住一个关键时序：会话联动 SQL 的入口在人工确认点，也就是 Query 处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且 Server Seed 可用时；不是“看完结果后再进入 SQL 工作台”。

### 3.1 功能全景

&emsp;&emsp;HermesAnalytics 数据分析师共有三个工作台：分析工作台、SQL 工作台、归因工作台。本课重点操作的是分析与 SQL 工作台：分析工作台负责自然语言问数，SQL 工作台负责独立 SQL 或来源查询的参数调整。归因工作台拿已完成的分析当证据运行多 Agent 辩论，本课只做全貌介绍，不展开操作。数据上下文位于分析页面，用来查看本次查询涉及的指标、维度、字段和治理信息，不是独立工作台。

&emsp;&emsp;会话联动 SQL 的入口是处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且取得 Server Seed 的待确认 Query，不是已经完成的结果。分析工作台在人工确认点有两种合法选择：直接确认执行；或满足资格时去会话联动 SQL 调整参数。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170244601.png" alt="本课重点的两个工作台" width="85%"></p>

### 3.2 分析工作台

1. 新建会话，或从左侧选择已有会话。
2. 在输入框中提交业务问题。
3. 查看规划、生成查询和安全校验等状态。
4. 打开确认卡，核对查询与参数，停在人工确认点，不先确认执行。
5. 在人工确认点二选一：路径 A 直接确认执行；或路径 B 当 Query 处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且 Server Seed 可用时，进入会话联动 SQL 查看参数边界、校验并受控执行。
6. 路径 A 观察结果表、结论、技术详情和右侧数据上下文；路径 B 完成 validation/execution 并受控回流后，再观察新结果与证据。

&emsp;&emsp;分析工作台是业务问题的入口，不提供自由编辑 SQL 的主要操作区。

&emsp;&emsp;这张分析工作台截图来自 HermesAnalytics 实际界面。看什么：左侧分析、SQL、归因三个工作台入口与会话导航，中间的新会话输入区和右侧数据上下文。能证明什么：HermesAnalytics 分析工作台可以真实打开并新建会话；当前画面尚未提交主案例，不能证明规划、SQL 执行或结果已经产生。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170534649.png" alt="分析工作台" width="95%"></p>

### 3.3 SQL 工作台

&emsp;&emsp;SQL 工作台有三种来源：自由 SQL、模板 SQL 和会话联动 SQL。本课重点比较自由 SQL 与会话联动 SQL；模板 SQL 只作全貌介绍，不展开操作。

&emsp;&emsp;自由 SQL 由分析师独立输入，经过校验后可以执行，但不能返回 Hermes 作为分析会话的可信结果。会话联动 SQL 只能从处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且取得 Server Seed 的待确认 Query 进入；SQL 结构只读，只能修改页面明确标记为可编辑的参数，不能绕过结构限制。模板 SQL 是分析师可保存、可复用的 SQL 模板，来源标识为 `ANALYST_TEMPLATE`（分析师模板 SQL），同样不能作为分析会话的可信结果返回。

&emsp;&emsp;这张自由 SQL 截图来自 HermesAnalytics 实际界面。看什么：SQL 编辑器、参数区、校验与执行入口。能证明什么：自由 SQL 可以独立编写、校验和执行，但页面没有“交给 Hermes 解析”的返回入口；这是自由 SQL 不进入分析会话可信结果的直观证据。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170235210.png" alt="自由 SQL 工作台" width="95%"></p>

### 3.4 三种 SQL 来源

<p style="text-align: center;"><strong>表 3-1　SQL 工作台三种来源的页面差异</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 对比项 | 自由 SQL | 模板 SQL | 会话联动 SQL |
| --- | --- | --- | --- |
| 来源标识 | `ANALYST_AD_HOC`（分析师自由 SQL） | `ANALYST_TEMPLATE`（分析师模板 SQL） | `CONVERSATION_DERIVED`（会话派生 SQL） |
| 入口 | 在 SQL 工作台独立新建 | 从模板列表选择 | 从处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且取得 Server Seed 的待确认 Query 进入 |
| SQL 编辑 | 可以编辑，仍需通过校验 | 可以编辑，是模板本体 | SQL 结构只读 |
| 参数调整 | 由分析师自行编写 | 在模板中定义 | 只改页面标记为可编辑的参数 |
| 执行 | 校验后执行 | 校验后执行 | 修改后重新校验、再执行 |
| 返回 Hermes | 不可以 | 不可以 | 符合条件时返回分析工作台 |

</div>

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170247577.png" alt="工作台双路径" width="85%"></p>

### 3.5 页面操作

&emsp;&emsp;分析工作台的操作顺序是：选择会话 → 输入问题 → 看状态 → 核对确认卡 → 停在人工确认点。此处只有两种合法选择：直接确认执行并查看结果与数据上下文；或当 Query 处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且 Server Seed 可用时，进入会话联动 SQL 查看参数边界、保留原值或仅在批准可编辑时修改、校验执行后再受控返回。不能先得到完成结果，再以该结果作为进入 SQL 工作台的入口。

&emsp;&emsp;会话联动 SQL 的页面顺序是：在待确认 Query + Server Seed 可用时进入 → 查看页面参数与批准边界 → 保留原值或仅在页面明确批准可编辑时修改参数 → 重新校验 → 执行 → 以 `execution_id + source_query_id` 返回分析工作台。页面没有可编辑参数时，不应自行假设或改写 SQL 结构。

### 3.6 源码入口

&emsp;&emsp;前端路由入口是 [`frontend/src/app/router.tsx`](../../HermesAnalytics/frontend/src/app/router.tsx)；分析页面入口是 [`frontend/src/features/analysis/pages/AnalystWorkspacePage.tsx`](../../HermesAnalytics/frontend/src/features/analysis/pages/AnalystWorkspacePage.tsx)；SQL 工作台入口是 [`frontend/src/features/workbench/pages/SqlWorkbenchPage.tsx`](../../HermesAnalytics/frontend/src/features/workbench/pages/SqlWorkbenchPage.tsx)。本章只用它们确认页面入口与操作顺序。

## <center>第四章：主案例的业务数据</center>

> 请分析 2026 年 6 月各渠道的净销售额，并与 2026 年 5 月比较。

&emsp;&emsp;这个需求包含四个业务条件：

- **净销售额** 作为核心指标，需要从两个治理视图取得金额并相减。
- **各渠道** 是分组维度，两条数据线需要先分别聚合再对齐。
- **2026 年 6 月** 是本期时间窗口，用半开区间表示。
- **与 2026 年 5 月比较** 引入一个对比期，最终要输出变化量和变化率。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170240203.png" alt="主案例从治理数据到净销售额对比" width="95%"></p>

### 4.1 净销售额：两本账和一个公式

&emsp;&emsp;主案例的净销售额由两笔金额相减得到：支付侧的实付商品金额，减去退款侧的退款商品金额。可以把这两个来源看成两本独立账本——支付账记录已支付明细的实付商品金额，退款账记录成功退款的退款商品金额。两个数据源分别按 `channel_id` 聚合，并在本期、对比期两个半开窗口中计算金额，再按渠道对齐相减，得到最终的净销售额。

&emsp;&emsp;下面三条规则是判断 SQL 正误的第一道门：

- 净销售额不是 `orders.order_amount`（订单实付商品金额）。金额来源是支付明细的 `paid_amount`（实付商品金额）与退款明细的 `refunded_amount`（退款商品金额）。
- `analytics.v_paid_order_items`（已支付订单明细视图）与 `analytics.v_refund_items`（成功退款明细视图）不能逐行连接后再求和，必须各自先按渠道聚合，再在渠道粒度相减。
- 退款属于哪个月份，看 `refunded_at`（退款成功时间），不按 `original_paid_at`（原订单支付时间）归属。

&emsp;&emsp;主案例只涉及电商业务里最窄的一条血缘，底层关系可以用三句话概括：

- `channels.channel_id` → `orders.channel_id`：订单挂在哪个渠道下。
- `orders.order_id` → `order_items.order_id`：一个订单包含多条明细。
- `order_items.order_item_id` → `refund_items.order_item_id`：退款精确对应到原支付明细。

&emsp;&emsp;`products` 与 `regions` 虽然参与了治理视图的完整构造，但主案例既不按商品分组也不按地区分组，因此不纳入查询。最终的查询只读取 `analytics` 治理视图，不直接读取 `ecommerce_core` 原始表。

<p style="text-align: center;"><strong>表 4-1　主案例的数据对象与关键字段</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 数据对象 | 每行代表什么 | 主案例关键字段 | 本案例用途 |
| --- | --- | --- | --- |
| `analytics.v_paid_order_items`（已支付订单明细视图） | 一条已支付订单明细 | `channel_id`（渠道ID）、`channel_name`（渠道名称）、`paid_amount`（实付商品金额）、`paid_at`（支付时间） | 支付侧聚合；视图已过滤 `PAID`（已支付）订单 |
| `analytics.v_refund_items`（成功退款明细视图） | 一条成功退款明细 | `channel_id`（渠道ID）、`channel_name`（渠道名称）、`refunded_amount`（退款商品金额）、`refunded_at`（退款成功时间） | 退款侧聚合；视图已过滤 `SUCCESS`（成功）退款 |
| `analytics.v_channels`（渠道目录视图） | 一个渠道 | `channel_id`（渠道ID）、`channel_name`（渠道名称）、`is_active`（是否启用） | 提供渠道目录、渠道名称和状态信息；参数校验从该治理目录读取渠道 ID；不是金额来源 |

</div>

### 4.2 各渠道：分别聚合，再按渠道对齐

&emsp;&emsp;各渠道作为分组维度，要求最终结果按渠道逐行输出。处理路径如下：

- 支付侧从 `v_paid_order_items` 按 `channel_id` 聚合 `paid_amount`；
- 退款侧从 `v_refund_items` 按 `channel_id` 聚合 `refunded_amount`；
- 两个聚合结果再按 `channel_id` 对齐。对齐模式采用 `anchor_left`——以支付侧为锚点，左连接退款侧。如果某个渠道没有退款，退款金额按 0 处理；不能因为缺少退款就把该渠道从结果中剔除。

&emsp;&emsp;渠道目录视图 `v_channels` 只提供渠道名称、ID 和状态，不参与金额计算。

&emsp;&emsp;常见的错误是把两个明细视图先按 `order_item_id` 连接，再直接求和。这类逐行连接可能造成重复匹配与金额膨胀，和独立聚合再对齐的做法完全不同。

### 4.3 6 月与 5 月：两个半开时间窗口

&emsp;&emsp;“2026 年 6 月”在 SQL 中不是 6 月 1 日到 6 月 30 日的闭区间，而是“包含开始、不包含结束”的半开区间。同样，5 月对比期也是半开区间。半开窗口会落到每个数据源的时间字段上：支付侧用 `paid_at`（支付时间），退款侧用 `refunded_at`（退款成功时间）。QueryPlanner 生成四个时间参数后，真正落到 SQL 里的过滤逻辑是：

- 支付侧当前期：`paid_at >= current_start AND paid_at < current_end`
- 退款侧当前期：`refunded_at >= current_start AND refunded_at < current_end`
- 支付侧对比期：`paid_at >= previous_start AND paid_at < previous_end`
- 退款侧对比期：`refunded_at >= previous_start AND refunded_at < previous_end`

&emsp;&emsp;这里的 `>=` 与 `<` 才是 SQL 比较运算；`current_start`、`current_end`、`previous_start`、`previous_end` 只是被赋值的日期参数。本案例的取值如下：

- `current_start`：2026-06-01
- `current_end`：2026-07-01
- `previous_start`：2026-05-01
- `previous_end`：2026-06-01

&emsp;&emsp;从源码看，`previous_start` 与 `previous_end` 由比较关系推导，并固定为不可编辑（`editable=False`）；`current_start` 与 `current_end` 继承 Manifest 中 `period_start`、`period_end` 的 `editable` 配置，当前案例中为 `true`。这是规划源码层的可编辑性；页面是否可编辑，还要以当前 Query 的 parameter contract（参数契约）和实际界面为准。

&emsp;&emsp;自然语言里的“6 月”到 6 月 30 日止，窗口结束参数取到 7 月 1 日（不包含），两者并不矛盾——前者是业务意图，后者是半开区间的执行边界。

### 4.4 跨期退款：退款算在哪个月

&emsp;&emsp;下面的手算示例使用教学虚构数据，演示“退款按退款成功时间归属”对净销售额的影响：

<p style="text-align: center;"><strong>表 4-2　跨期退款教学手算示例（教学虚构数据）</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 月份 | 实付商品金额 | 按退款发生时间归属的退款 | 净销售额 |
| --- | ---: | ---: | ---: |
| 2026 年 5 月 | 800 | 50 | 750 |
| 2026 年 6 月 | 1000 | 100 | 900 |

</div>

&emsp;&emsp;6 月的 100 元退款可以来自一笔 5 月支付的订单，但因为它在 6 月退款成功，业务口径仍把它扣在 6 月。于是变化量 = 900 − 750 = 150，变化率 = 150 ÷ 750 = 20%。

### 4.5 主案例业务口径核对清单

<p style="text-align: center;"><strong>表 4-3　主案例业务口径核对清单</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 检查项 | 看到什么算通过 | 常见错误 |
| --- | --- | --- |
| ① 指标公式 | 净销售额 = `SUM(paid_amount)` − `SUM(refunded_amount)`，分别从 `v_paid_order_items` 和 `v_refund_items` 取得 | 直接使用 `orders.order_amount` 或单一视图计算 |
| ② 支付侧数据与时间字段 | 使用 `paid_amount` 和 `paid_at` | 误用 `order_amount`、`business_date` 或其他字段 |
| ③ 退款侧数据与时间字段 | 使用 `refunded_amount` 和 `refunded_at` | 按 `original_paid_at` 归属退款，或使用错误的退款金额字段 |
| ④ 两个半开窗口 | 支付侧按 `paid_at`、退款侧按 `refunded_at`，分别使用当前期 `[current_start, current_end)` 与对比期 `[previous_start, previous_end)`，端点不重不漏 | 两侧使用了错误的时间字段，或使用 `BETWEEN` 闭区间把窗口结束点的数据包含进来 |
| ⑤ 渠道聚合与对齐 | 两侧先按 `channel_id` 聚合，再用 `LEFT JOIN` 以支付侧为锚点对齐，退款缺失补 0 | 两个明细视图先逐行连接后再聚合，或退款缺失保留 NULL |
| ⑥ 结果列与除零保护 | 输出 `current_value`、`previous_value`、`change_amount`、`change_rate`；`change_rate` 使用 `NULLIF(previous_value, 0)` 防止除零 | 缺列，或直接用 `previous_value` 做分母而未处理 0 |

</div>

### 4.6 去源码核对业务口径

&emsp;&emsp;以下源码入口可直接验证本章的各项结论：

- 打开 [`manifest.json`](../../HermesAnalytics/database/domain/default-ecommerce/manifest.json) 的 `net_sales`（净销售额）指标，确认公式 `sum_paid_amount - sum_refunded_amount`、数据源 `analytics.v_paid_order_items` 与 `analytics.v_refund_items`、对齐方式 `anchor_left` 以及缺失值 `zero`。
- 打开 [`20260731_0001_initial_schema.py`](../../HermesAnalytics/database/migrations/versions/20260731_0001_initial_schema.py)，查看两个治理视图的过滤条件与时间字段 `paid_at`、`refunded_at`。
- 打开 [`planner.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/planning/planner.py) 的 `_time_parameters`（时间参数），确认四个时间参数与 `previous_*` 固定不可编辑的推导逻辑；打开 [`compiler.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/compilation/compiler.py) 的 `_compile_multi_source_comparison`（多源对比编译）、`_source_branches`（源分支）与 `_aligned_sources`（对齐后的源），确认多源聚合、渠道对齐和结果列。

&emsp;&emsp;运行时判断以数据库当前激活的 active domain version（活动领域版本）为准；源码用于理解业务口径，最终查询还需要结合当前激活版本核对。

&emsp;&emsp;到这里，你已经能从原句拆出四个业务条件，再用表 4-3 的核对清单逐项检查查询：指标公式、两侧数据与时间字段、两个半开窗口、渠道聚合与对齐、结果列与除零保护。

## <center>第五章：完整问数</center>

&emsp;&emsp;你已经从主案例原句确认了指标、维度、时间窗口和对比口径。现在回到分析工作台，沿同一个问题走完提交、人工确认、直接执行和条件式联动两条路径。每到确认卡，就用第四章表 4-3 的核对清单逐项核对查询是否符合业务口径。

> **证据边界**：本章以 HermesAnalytics 实际界面为准。历史结果形态只作参照，不能替换当前实测；历史记录属于历史证据，不能冒充 HermesAnalytics 当前运行结果。你的本机状态以 HermesAnalytics 重新实测为准。

### 5.1 输入主案例

&emsp;&emsp;在分析工作台新建或选择会话，输入本课唯一问题：

> **请分析 2026 年 6 月各渠道的净销售额，并与 2026 年 5 月比较。**

&emsp;&emsp;提交后观察 `AGENT_PLANNING`（理解问题）、`COMPILING`（生成查询）、`POLICY_CHECKING`（安全校验）。这三个状态表示系统正在理解问题、生成查询和执行安全校验，不表示 SQL 已经执行。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170247889.png" alt="主案例状态" width="85%"></p>

&emsp;&emsp;**源码定位**：[`turns.py`](../../HermesAnalytics/backend/src/hermes_analytics/api/routes/turns.py) 的 `submit_turn` 接收问题，`turn_events` 返回状态事件；完整入口见表 5-2。

### 5.2 查看确认卡

&emsp;&emsp;在 `POLICY_CHECKING`（安全校验）后打开确认区域，查看 FrozenQuery、参数和 `plan_hash`。此时页面应停在 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）人工确认点；确认前不要把状态变化误读成业务 SQL 已执行。Query 已停在人工确认点；是否进入会话联动 SQL，必须在点击直接确认执行之前决定；没有 Server Seed 或 Seed 已过期时，不能创建联动种子。

&emsp;&emsp;核对 FrozenQuery（冻结查询）与 SQL 时，按第四章表 4-3 的六项检查清单逐项核对数据源、时间字段、独立聚合和渠道对齐。

&emsp;&emsp;这张人工确认卡截图来自 HermesAnalytics 实际界面。看什么：确认卡上的 SQL、参数和 `plan_hash`，以及“确认执行”按钮。能证明什么：HermesAnalytics 确实在程序生成冻结内容后停在人工确认点，模型手里没有请求执行工具；是否进入会话联动 SQL，必须在点击确认执行之前决定。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170302619.png" alt="人工确认卡" width="95%"></p>

&emsp;&emsp;**源码定位**：[`bridge.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_planning/bridge.py) 的 `park_ready_for_confirmation` 把 Query 停在人工确认点；完整校验、规划与编译入口见表 5-2。

### 5.3 确认并执行

&emsp;&emsp;核对确认卡后，在人工确认点选择路径 A：直接确认执行，继续观察 `EXECUTING`（执行查询）、`RESULT_VALIDATING`（校验结果）、`INTERPRETING`（解读结果）。如果页面显示拒绝或校验失败，应保留失败状态，不把它写成执行成功。

&emsp;&emsp;**源码定位**：[`analysis_execution/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/service.py) 的 `confirm` 接收确认，随后由 `_execute_confirmed_queries` 进入执行链。

### 5.4 查看结果

&emsp;&emsp;路径 A 执行完成后同时查看结果表、结论、技术详情、数据上下文、ResultSnapshot 和 FactRef。页面上的结果与解读来自本次执行形成的结果对象和证据对象；这些对象怎样限制解读范围，在第六章 6.6 继续分析。

&emsp;&emsp;这张路径 A 结果截图来自 HermesAnalytics 实际界面。看什么：结果表、结论、技术详情和数据上下文。能证明什么：HermesAnalytics 在人工确认后完成了执行、结果校验和受控解读，页面能同时看到结果与证据；不能证明未来任何环境都会成功。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170303148.png" alt="路径 A 结果" width="95%"></p>

<p style="text-align: center;"><strong>表 5-1　路径 A（直接确认执行）的历史结果形态</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 项目 | 历史已证实信息 | 本课判断 |
| --- | --- | --- |
| QueryRun | 1 个 | 不是 6 月和 5 月各一条 QueryRun |
| 行数 | 6 行 | 不据此编造金额 |
| 列 | `channel_id`、`channel_name`、`current_value`、`previous_value`、`change_amount`、`change_rate` | 表示渠道与双周期比较形态 |
| assistant answer | 存在 | 解读是结果对象的一部分 |
| `fact_refs` | 28 个 | 事实引用来自服务端证据 |
| `result_snapshot_id` | 存在 | 结果可定位到服务端快照 |

</div>

&emsp;&emsp;这张表是路径 A（直接确认执行）的历史 API 真栈结果形态，属于历史证据，不能作为当前浏览器实测的替代。当前浏览器路径 A 以 HermesAnalytics 实际界面为准；你的本机状态以 HermesAnalytics 重新实测为准。

&emsp;&emsp;列形态只说明输出结构，SQL 是否正确还要按第四章表 4-3 核对数据源、时间字段、独立聚合和渠道对齐。

&emsp;&emsp;**源码定位**：[`analysis_execution/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/service.py) 按 `validator → evidence → interpreter` 形成结果与解读。这里先确认结果形成顺序；可信边界见第六章 6.6。

### 5.5 多轮追问

&emsp;&emsp;分析工作台支持在同一会话里继续追问，并继承上一轮上下文。追问也需要重新经过规划、生成、Policy、人工确认等链路，不能因为“已经问过一次”就跳过确认。

&emsp;&emsp;HermesAnalytics 实际界面已真实跑通一次追问：在路径 A 结果之后追问“哪个渠道下降最多？”，再次确认并完成解读，结果为“抖音”。这张追问结果截图能证明：同一会话内上下文继承、再次确认、再次执行与再解读的链路在当前环境可用。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170303236.png" alt="多轮追问结果" width="95%"></p>

&emsp;&emsp;**源码定位**：[`AnalystConversationWorkspace.tsx`](../../HermesAnalytics/frontend/src/features/analysis/components/AnalystConversationWorkspace.tsx) 的 `submitTurn` 同时承接首次提问与后续追问，复用同一提交主链。

### 5.6 进入 SQL 工作台

&emsp;&emsp;路径 B 仍从人工确认点开始：只有该来源 Query 处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且 Server Seed 有效时，才从待确认 Query 进入会话联动 SQL 工作台。先确认页面显示来源信息和当前允许修改的参数，不要切换到自由 SQL，也不要另造第二个业务问题。如果 Query 已完成、Seed 已过期或页面无有效 Seed，则不能创建联动种子，应停在当前路径。

&emsp;&emsp;这张联动 SQL 入口截图来自 HermesAnalytics 实际界面。看什么：从待确认 Query 进入 SQL 工作台后的来源信息、SQL 结构与参数区。能证明什么：HermesAnalytics 在人工确认点提供了会话联动 SQL 入口，且 SQL 结构只读；不能证明可以自由修改 SQL 结构。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170306395.png" alt="联动 SQL 入口" width="95%"></p>

&emsp;&emsp;**源码定位**：[`analysis_execution/queries.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/queries.py) 的 `QueryProjectionService.workbench_seed` 返回可用的联动 Seed。

### 5.7 校验并执行

&emsp;&emsp;路径 B 中，以当前主案例的当前页面契约为准：主案例 `comparison=previous_period` 会生成 `current_start`、`current_end`、`previous_start`、`previous_end` 四个日期参数；当前 parameter contract（参数契约）中这 4 项在联动 SQL 页面全部显示“查询计划已锁定此参数”，`editable=0`。这是当前主案例的锁定表现，不代表所有合法 Query 都不可编辑。保留原值 → 校验 → 运行 → 查看临时结果，是当前页面允许的操作。

&emsp;&emsp;如果未来或其他合法 Query 确有页面批准的可编辑参数，才选择其中一项修改，然后按页面顺序重新校验、执行；修改后，旧的校验与执行结果不再代表当前页面输入。页面必须提交 `source_query_id + seed_id + parameter_values` 做 validation，再用 `validation_id + plan_hash` 执行；出现新的校验结果和新的执行结果后，才能继续返回分析工作台。

&emsp;&emsp;这两张截图来自 HermesAnalytics 实际界面。第一张看当前主案例的四个锁定参数：`current_start`、`current_end`、`previous_start`、`previous_end` 都标记为“查询计划已锁定此参数”，`editable=0`。能证明什么：当前主案例在联动 SQL 里不能现场改日期或 SQL，这不是故障，而是当前参数契约锁定的表现。第二张看校验与执行：保留原值后 validation 通过、execution 成功，页面出现新的校验结果和执行结果。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170620202.png" alt="四个锁定参数" width="95%"></p>

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170308532.png" alt="联动校验与执行" width="95%"></p>

&emsp;&emsp;**源码定位**：[`application/workbench/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/workbench/service.py) 的 `validate` 先完成校验，再由 `execute` 执行已校验请求。

### 5.8 返回分析工作台

&emsp;&emsp;路径 B 执行成功后，以 `execution_id + source_query_id` 返回分析工作台；前端请求同时携带 `expected_turn_version` 和 `client_request_id`。服务端接收这些引用后恢复分析会话，并继续形成新的结果与解读；具体复核边界在第六章 6.7 展开。

&emsp;&emsp;这张回流截图来自 HermesAnalytics 实际界面。看什么：执行成功后回到分析工作台，页面展示新的结果与解读。能证明什么：HermesAnalytics 的 `execution_id + source_query_id` 回流链路已真实走通；回流后的复核规则与证据边界以第六章 6.7 为准。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170310475.png" alt="联动回流结果" width="95%"></p>

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170251725.png" alt="跨平台问数" width="85%"></p>

&emsp;&emsp;**源码定位**：[`routes/turns.py`](../../HermesAnalytics/backend/src/hermes_analytics/api/routes/turns.py) 的 `resume_from_workbench` 接收工作台执行引用并恢复分析会话。

### 5.9 第五章源码导航

&emsp;&emsp;第五章每个页面动作都对应一条源码调用链。按共用链路、路径 A、路径 B 三段定位最小源码入口；阅读时先沿页面动作找到对应函数，不需要逐行读整个项目。在本章先看调用关系；Policy、Freeze、SafeExecutor、FactRef 的可信边界放到第六章继续分析。

<p style="text-align: center;"><strong>表 5-2　共用链路：从提交到人工确认</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 环节 | 源码入口 | 页面动作对应什么 |
| --- | --- | --- |
| 页面提交与事件流 | 前端 [`AnalystConversationWorkspace.tsx`](../../HermesAnalytics/frontend/src/features/analysis/components/AnalystConversationWorkspace.tsx)（`submitTurn` 定位 1470 行、`streamTurnEvents` 定位 1118 行）；网关 [`HttpAnalyticsGateway.ts`](../../HermesAnalytics/frontend/src/shared/gateway/HttpAnalyticsGateway.ts)（`submitTurn` 定位 742 行、`streamTurnEvents` 定位 755 行） | 页面调用 Gateway，再由 Gateway 发 HTTP 请求；状态来自 SSE 事件流 |
| 后端入口与产品阶段 | 后端 [`turns.py`](../../HermesAnalytics/backend/src/hermes_analytics/api/routes/turns.py)（`submit_turn` 定位 42 行、`turn_events` 定位 97 行）；[`stages.py`](../../HermesAnalytics/backend/src/hermes_analytics/domain/stages.py)（`ProductStage` 定位 16 行） | 提交后进入理解问题、生成查询、安全校验三个阶段；这三个状态不代表 SQL 已执行 |
| Hermes 工具面 | 后端 [`hermes_adapter/plugin.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/plugin.py)（`submit_query_plan` 定位 195 行；工具清单中没有 `execute_sql`） | 模型提交结构化 Intent，不能直接请求执行 SQL |
| 规划与冻结 | 后端 [`analysis_planning/bridge.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_planning/bridge.py)（`validate_submission` 定位 249 行、`plan_queries` 定位 328 行、`compile_queries` 定位 353 行、`inspect_policy` 定位 377 行、`park_ready_for_confirmation` 定位 697 行）；[`intent/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/intent/service.py)（`IntentValidator` 定位 94 行） | 按顺序走校验、规划、编译、策略检查，最后停在人工确认点 |
| 策略与冻结 | 后端 [`policy/inspector.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/policy/inspector.py)（`SqlPolicy.inspect` 定位 22 行）；[`compilation/freeze.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/compilation/freeze.py)（`FrozenQueryFactory.freeze` 定位 25 行） | 策略检查只做确定性校验；Freeze 生成 FrozenQuery 与 `plan_hash`，不执行数据库 |

</div>

<p style="text-align: center;"><strong>表 5-3　路径 A：确认执行与结果返回</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 环节 | 源码入口 | 页面动作对应什么 |
| --- | --- | --- |
| 确认与执行 | 后端 [`turns.py`](../../HermesAnalytics/backend/src/hermes_analytics/api/routes/turns.py)（`confirm_turn_execution` 定位 139 行）；[`analysis_execution/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/service.py)（`confirm` 定位 105 行、`_execute_confirmed_queries` 定位 250 行） | 页面确认进入执行服务，再转入已确认查询的执行流程 |
| 只读执行器 | 后端 [`safe_executor.py`](../../HermesAnalytics/backend/src/hermes_analytics/infrastructure/postgres/safe_executor.py)（`execute` 定位 46 行、`_verify_gate` 定位 99 行） | 执行服务调用 SafeExecutor 完成数据库查询；可信边界见 6.6 |
| 结果形成 | 后端 [`analysis_execution/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/service.py)（校验调用点定位 346 行、结果对象调用点定位 347 行、解读调用点定位 530 行） | 执行结果依次进入校验、结果对象形成和解读入口 |
| 结果返回页面 | 后端 [`hermes_interpreter.py`](../../HermesAnalytics/backend/src/hermes_analytics/infrastructure/hermes_interpreter.py)（`HermesResultInterpreter.interpret` 定位 185 行） | 解读完成后返回结果页面；证据对象的可信边界见 6.6 |

</div>

<p style="text-align: center;"><strong>表 5-4　路径 B：联动 SQL、校验执行与会话回流</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 环节 | 源码入口 | 页面动作对应什么 |
| --- | --- | --- |
| Server Seed 资格门 | 后端 [`routes/queries.py`](../../HermesAnalytics/backend/src/hermes_analytics/api/routes/queries.py)（`workbench_seed` 定位 80 行）；[`analysis_execution/queries.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/queries.py)（`workbench_seed` 定位 38 行） | 只有 `AWAITING_EXECUTION_CONFIRMATION` 且有有效 Seed 才能进入联动 SQL |
| 联动 SQL 页面 | 前端 [`SqlWorkbenchPage.tsx`](../../HermesAnalytics/frontend/src/features/workbench/pages/SqlWorkbenchPage.tsx)（`getAnalystQueryDetail + getWorkbenchSeed` 定位 509-511 行）；[`SqlWorkbench.tsx`](../../HermesAnalytics/frontend/src/features/workbench/components/SqlWorkbench.tsx)（`validate` 定位 202 行、`executeValidation` 定位 255 行） | 页面同时加载 Query 详情与 Seed；校验、执行都从组件方法进入 |
| 后端校验与执行 | 后端 [`routes/workbench.py`](../../HermesAnalytics/backend/src/hermes_analytics/api/routes/workbench.py)（`create_validation` 定位 102 行、`create_execution` 定位 127 行）；[`application/workbench/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/workbench/service.py)（`validate` 定位 58 行、`execute` 定位 376 行） | 页面先创建 validation，再以 `validation_id + plan_hash` 创建 execution |
| 回流 | 后端 [`routes/turns.py`](../../HermesAnalytics/backend/src/hermes_analytics/api/routes/turns.py)（`resume_from_workbench` 定位 203 行）；[`analysis_execution/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/service.py)（`resume_from_workbench` 定位 144 行） | 回流提交 `execution_id + source_query_id + expected_turn_version + client_request_id` |
| 会话恢复与新结果形成 | 后端 [`analysis_execution/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/service.py)（`resume_from_workbench` 定位 144 行、工作台结果解读入口定位 192 行） | 回流后恢复分析会话并形成新的结果与解读；复核边界见 6.7 |

</div>

## <center>第六章：问数为什么可信</center>

&emsp;&emsp;上一章已经说明主案例的合法操作路径，现在我们进入“问数为什么可信”。可信来自模型、程序、人、执行器和数据库各自只做被允许的事，而不是相信模型会自动遵守要求。

&emsp;&emsp;我们先把最容易产生的四个误解拆掉，再进入具体机制：

<p style="text-align: center;"><strong>表 6-1　常见误解与真实机制</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 常见误解 | 它不是什么 | 它是什么 | 判断边界 | 源码锚点 | 生活化类比 | 反例 |
| --- | --- | --- | --- | --- | --- | --- |
| 模型会不会自由写 SQL？ | 模型不是“会写 SQL 的助手” | 在问数规划阶段，模型提交结构化 Intent；在解读阶段，模型另提交结构化答案和 FactRef 选择 | 模型手里没有 `execute_sql` 工具 | [`plugin.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/plugin.py) | 点菜的人不自煮菜 | 模型直接生成并执行 SQL，属于越权 |
| `plan_hash` 是否等于人工确认？ | `plan_hash` 不是执行许可 | 它只标识冻结内容，人确认才是执行许可 | 没有人工确认，即使 hash 相同也不能执行 | [`bridge.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_planning/bridge.py) | 合同编号不等于签字 | 看到 hash 相同就执行，绕过确认 |
| 联动 SQL 能否只凭 `source_query_id` 返回？ | `source_query_id` 不是万能钥匙 | 必须同时有 `execution_id`，服务端复核来源与血缘 | 缺少 `execution_id` 或被复核拒绝，不得返回 | [`analysis_execution_repository.py`](../../HermesAnalytics/backend/src/hermes_analytics/infrastructure/postgres/analysis_execution_repository.py)，定位 `lineage_valid` | 出差报销要有票根 | 只传来源编号就当作结果已回流 |
| Manifest 文件改完是否立即生效？ | 文件不是运行时开关 | 需要走发布/激活流程，Active DomainCatalog 才被程序使用 | 文件存在不等于当前版本已激活 | [`publish.py`](../../HermesAnalytics/database/hermes_analytics_database/publish.py)，定位 `publish_manifest`/`active_domain_version_id` | 菜谱入库不等于大厨已采用 | 直接改 Manifest 后认为 SQL 已按新口径生成 |

</div>

&emsp;&emsp;这四个误解的共同点，是把“看起来像”当成“实际上是”。真实机制的关键是看谁有权跨过哪一条边界：模型不能自己执行 SQL，`plan_hash` 不能替代人确认，联动返回必须经服务端复核，Manifest 文件必须经过运行时激活才成为业务真值。

### 6.1 六个状态

&emsp;&emsp;完整状态链是：`AGENT_PLANNING`（理解问题）→ `COMPILING`（生成查询）→ `POLICY_CHECKING`（安全校验）→ 人工确认 → `EXECUTING`（执行查询）→ `RESULT_VALIDATING`（校验结果）→ `INTERPRETING`（解读结果）。人工确认是独立暂停点，不是模型状态；Freeze 位于 `POLICY_CHECKING`（安全校验）内。

<p style="text-align: center;"><strong>表 6-2　可信问数状态链</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 阶段或门 | 主要动作 | 判断边界 |
| --- | --- | --- |
| `AGENT_PLANNING`（理解问题） | 模型提交结构化 Intent | 不生成或执行 SQL |
| `COMPILING`（生成查询） | 程序校验并编译 SQL | 不回读原话补猜 |
| `POLICY_CHECKING`（安全校验） | Policy 检查并生成 FrozenQuery | 未通过不得确认 |
| 人工确认 | 人确认冻结内容 | 未确认不得执行 |
| `EXECUTING`（执行查询） | Safe Executor 执行 | 受确认、预算和只读边界限制 |
| `RESULT_VALIDATING`（校验结果） | 校验结果结构并生成证据 | 结果表不等于最终解释 |
| `INTERPRETING`（解读结果） | Hermes 基于证据解释 | 不自行构造事实 |

</div>

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170254712.png" alt="可信状态链" width="85%"></p>

&emsp;&emsp;源码证据入口是 [`backend/src/hermes_analytics/domain/stages.py`](../../HermesAnalytics/backend/src/hermes_analytics/domain/stages.py)，只核对 `ProductStage` 及显示映射；人工确认落在哪里，看 [`application/analysis_planning/bridge.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_planning/bridge.py) 的 `park_ready_for_confirmation`。

### 6.2 五类职责

<p style="text-align: center;"><strong>表 6-3　五类责任边界</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 责任方 | 负责什么 | 不能做什么 |
| --- | --- | --- |
| 模型 | 输出结构化 JSON、发起澄清、基于证据解释 | 不写 SQL、不执行 SQL、没有 `execute_sql` |
| 程序 | 校验 Intent、编译 SQL、做 Policy、冻结、校验结果和生成证据 | 不从自然语言自行补猜 |
| 人 | 确认 FrozenQuery | 不把确认变成模型自由执行许可 |
| Executor | 按冻结内容或工作台契约执行 | 不越过确认和参数边界 |
| 数据库 | 通过受限只读角色提供治理后的数据 | 不接受模型直连 |

</div>

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170254882.png" alt="五类职责" width="85%"></p>

### 6.3 Intent 与澄清

&emsp;&emsp;Intent 是模型提交的结构化分析意图。模型只输出结构化 JSON；如果时间、排名数量等信息缺失，程序返回 `INTENT_INCOMPLETE`（分析意图信息不完整）、`TIME_RANGE_REQUIRED`（缺少时间范围）等机器可读错误码，模型再把缺口翻译成人能理解的澄清问题。程序不自行补日期，也不直接生成中文问句。

&emsp;&emsp;决定性源码入口是 [`backend/src/hermes_analytics/hermes_adapter/plugin.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/plugin.py) 和 [`backend/src/hermes_analytics/nl2sql/intent/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/intent/service.py)。前者可核对结构化工具且不存在 `execute_sql`，后者可核对 Intent 校验和错误码。

### 6.4 编译 SQL

&emsp;&emsp;Compiler 只接收通过校验的 Intent 和 Manifest 中的业务定义，按确定规则形成 QueryPlan 和参数化 SQL。不能支持的指标、维度、比较或排名会受控失败，不存在“编译失败后让模型自由写 SQL”的第二通道。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170256266.png" alt="确定性编译" width="85%"></p>

&emsp;&emsp;源码从 [`backend/src/hermes_analytics/application/analysis_planning/bridge.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_planning/bridge.py) 进入编译与冻结路径。

### 6.5 检查并冻结

&emsp;&emsp;Policy 对编译后的 SQL 做确定性检查；通过后，Freeze 把 SQL、参数、版本和结果契约固定成 FrozenQuery，并生成 `plan_hash`。页面停在人工确认点，人确认的是这份完整冻结内容；任何相关内容变化都应重新检查和确认。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170259160.png" alt="检查与冻结" width="85%"></p>

&emsp;&emsp;源码证据入口是 [`backend/src/hermes_analytics/nl2sql/policy/inspector.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/policy/inspector.py)、[`backend/src/hermes_analytics/application/analysis_planning/bridge.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_planning/bridge.py) 和 [`backend/src/hermes_analytics/nl2sql/compilation/freeze.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/compilation/freeze.py)。

### 6.6 执行与证据

&emsp;&emsp;人工确认后，Safe Executor 核对确认状态和 `plan_hash`，再按只读与预算条件执行。`ResultStructureValidator` 按 result contract 中 `grain` 列组合检查唯一性，并检查结果列和限制状态；服务端持久化 ResultSnapshot，`EvidenceBuilder` 形成 ResultDigest 和 allowed FactRef；Hermes 只基于 ResultDigest 与 allowed FactRef 做受控解读。

<p style="text-align: center;"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260812170259583.png" alt="执行与证据" width="85%"></p>

&emsp;&emsp;源码证据入口是 [`backend/src/hermes_analytics/infrastructure/postgres/safe_executor.py`](../../HermesAnalytics/backend/src/hermes_analytics/infrastructure/postgres/safe_executor.py)、[`backend/src/hermes_analytics/application/analysis_execution/validation.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/validation.py) 和 [`backend/src/hermes_analytics/application/analysis_execution/evidence.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/evidence.py)。

### 6.7 工作台三个时点

&emsp;&emsp;会话联动 SQL 的真实时序分为三个时点：

- **时点 A**：Query 必须处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且 Server Seed 有效；已有 `queryId` 后，并行加载 Query 详情与 Server Seed；`source_query_id` 来自 Query 详情，`seed_id` 来自 Server Seed。已完成 Query 不能重新取得联动 Seed。
- **时点 B**：提交 `source_query_id + seed_id + parameter_values` 做 validation。参数变化后，旧 validation 和旧 execution 失效。
- **时点 C**：用 `validation_id + plan_hash` 执行；成功后提交 `execution_id + source_query_id` 返回分析工作台，前端请求还携带 `expected_turn_version` 与 `client_request_id`，由服务端复核来源、版本和幂等。

&emsp;&emsp;浏览器只提交受控引用和参数，不上传结果行作为证据。源码入口是 [`frontend/src/features/workbench/pages/SqlWorkbenchPage.tsx`](../../HermesAnalytics/frontend/src/features/workbench/pages/SqlWorkbenchPage.tsx) 和 [`frontend/src/features/workbench/components/SqlWorkbench.tsx`](../../HermesAnalytics/frontend/src/features/workbench/components/SqlWorkbench.tsx)。

## <center>第七章：总结与自测</center>

&emsp;&emsp;前六章依次完成安装、架构与业务地图、本课重点的两个工作台、主案例业务数据、两条合法问数路径和可信原理。最后我们用六项成果、主案例复述、源码索引和证据标签收束全课。

### 7.1 六项学习成果

<p style="text-align: center;"><strong>表 7-1　全课六项学习成果</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 序号 | 学习成果 | 通过标准 |
| --- | --- | --- |
| 1 | 安装证据 | 区分容器状态、readiness、模型可用和当前 E2E |
| 2 | 项目架构 | 说清四服务、Hermes 与前后端连接，并说清哪些由 Hermes 提供、哪些未启用或未暴露、哪些由 HermesAnalytics 自己负责 |
| 3 | 已有业务内容 | 说清 5 个指标、4 类维度、主案例净销售额口径，以及两个事实视图、双时间字段和先聚合后对齐 |
| 4 | 本课重点的两个工作台用法 | 完成分析工作台操作并区分自由 SQL 与会话联动 SQL |
| 5 | 跨平台完整问数 | 在人工确认点区分直接执行和条件式联动；联动需有效 Server Seed |
| 6 | 可信状态链 | 复述六个状态、人工确认和五类职责 |

</div>

### 7.2 复述主案例

&emsp;&emsp;提交问题前，先用第四章的数据标准复述主案例：支付侧使用 `analytics.v_paid_order_items`（已支付订单明细视图）的 `paid_at`（支付时间）与 `paid_amount`（实付商品金额），退款侧使用 `analytics.v_refund_items`（成功退款明细视图）的 `refunded_at`（退款成功时间）与 `refunded_amount`（退款商品金额）；两边分别在 6 月和 5 月两个半开时间窗口内按 `channel_id`（渠道ID）独立聚合，再以支付侧为主（`anchor_left`）对齐，退款缺失按 0 处理；最终检查 `current_value`（当前值）、`previous_value`（对比值）、`change_amount`（变化量）、`change_rate`（变化率）四个结果列，并确认 0 分母由 `NULLIF(previous_value, 0)` 处理。

&emsp;&emsp;请用唯一主案例复述：在分析工作台提交问题，查看确认卡并停在人工确认点。若选择路径 A，直接确认执行并得到结果与证据；若选择路径 B，仅当 Query 处于 `AWAITING_EXECUTION_CONFIRMATION`（等待确认执行）且 Server Seed 有效时，从待确认 Query 进入会话联动 SQL 工作台，保留原值或仅在页面批准可编辑时修改参数，重新校验和执行，最后以 `execution_id + source_query_id` 返回分析工作台（前端同时携带版本与幂等字段）。模型没有 `execute_sql`，Hermes 只基于服务端复核后的新证据解释。

### 7.3 源码索引

<p style="text-align: center;"><strong>表 7-2　源码导航索引</strong></p>

<style>.center { text-align: center; } .center table { margin-left: auto !important; margin-right: auto !important; } .center th, .center td { text-align: center !important; }</style>

<div class="center">

| 对应内容 | 源码入口 | 只看什么 | 判断问题 |
| --- | --- | --- | --- |
| 安装与启动 | [`scripts/bootstrap.sh`](../../HermesAnalytics/scripts/bootstrap.sh) | 启动、等待与 dry-run | 启动成功能证明什么？ |
| 可见状态 | [`domain/stages.py`](../../HermesAnalytics/backend/src/hermes_analytics/domain/stages.py) | `ProductStage` | 产品阶段有哪些？人工确认落在哪里？ |
| 模型工具 | [`hermes_adapter/plugin.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/plugin.py) | 结构化工具注册 | 为什么没有 `execute_sql`？ |
| Hermes 边界与上下文 | [`hermes_adapter/agent.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/agent.py) / [`context.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/context.py) / [`analysis_planning_repository.py`](../../HermesAnalytics/backend/src/hermes_analytics/infrastructure/postgres/analysis_planning_repository.py) / [`admin/plugin.py`](../../HermesAnalytics/backend/src/hermes_analytics/hermes_adapter/admin/plugin.py) | `skip_memory`、`build_explicit_history`、`LIMIT 6`、是否注册原生 skills | 项目多轮追问靠什么传回历史？`backend/skills` 是 Hermes 原生 skills 吗？ |
| Intent 与澄清 | [`intent/service.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/intent/service.py) | Intent 校验与错误码 | 谁判断缺口，谁生成问句？ |
| 编译、Policy 与冻结 | [`analysis_planning/bridge.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_planning/bridge.py) / [`freeze.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/compilation/freeze.py) / [`policy/inspector.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/policy/inspector.py) | `FrozenQuery`、`plan_hash`、Policy | 人确认的内容是什么？ |
| 执行与证据 | [`safe_executor.py`](../../HermesAnalytics/backend/src/hermes_analytics/infrastructure/postgres/safe_executor.py) / [`validation.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/validation.py) / [`evidence.py`](../../HermesAnalytics/backend/src/hermes_analytics/application/analysis_execution/evidence.py) | 执行门、结果校验、FactRef | 谁执行，谁生成证据？ |
| 工作台三个时点 | [`SqlWorkbenchPage.tsx`](../../HermesAnalytics/frontend/src/features/workbench/pages/SqlWorkbenchPage.tsx) / [`SqlWorkbench.tsx`](../../HermesAnalytics/frontend/src/features/workbench/components/SqlWorkbench.tsx) | 来源、校验、执行和返回 | 三个时点分别提交什么？ |
| 已有指标口径 | [`manifest.json`](../../HermesAnalytics/database/domain/default-ecommerce/manifest.json) | `net_sales` | 主案例净销售额口径是什么？ |
| 主案例时间窗口 | [`planner.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/planning/planner.py) | `_time_parameters` | 6 月与 5 月如何变成四个参数？ |
| 主案例双源 SQL | [`compiler.py`](../../HermesAnalytics/backend/src/hermes_analytics/nl2sql/compilation/compiler.py) | `_compile_multi_source_comparison` / `_aligned_sources` | 为什么先独立聚合再按渠道对齐？ |

</div>

### 7.4 证据标签

&emsp;&emsp;`本机实测` 表示刚刚在本机真实运行的结果，并需区分本机当前实测与历史实测；`源码核验` 表示源码符号存在；`尚未验证` 表示当前环境尚未取得足够证据。当前浏览器证据以 HermesAnalytics 实际界面为准；历史记录只作参照，不能替代 HermesAnalytics 当前结果。